# NIH14 Concept Contribution Ranking

Aggregate concept activations multiplied by classification weights to reproduce the concept contribution tables per pathology.


In [ ]:
import os
import json
from pathlib import Path

import torch
import numpy as np

os.chdir('..')

import cbm
import data_utils
from evaluate_cbm_nih import _load_thresholds


In [ ]:
MODEL_DIR = 'saved_models/nih14_example'
DEVICE = 'cuda'
CLASS_NAME = 'Pneumonia'
THRESHOLDS_PATH = 'saved_models/nih14_example/thresholds.json'
THRESHOLD_FALLBACK = 0.5


In [ ]:
model = cbm.load_cbm(MODEL_DIR, DEVICE)
model.eval()

with open(Path(MODEL_DIR) / 'args.txt') as f:
    args = json.load(f)
classes_path = data_utils.LABEL_FILES['nih14']
with open(classes_path) as f:
    classes = [c for c in f.read().splitlines() if c]
class_idx = classes.index(CLASS_NAME)
print(f'Loaded model covering {len(classes)} pathologies. Editing class {CLASS_NAME} (index {class_idx}).')

try:
    thresholds = _load_thresholds(THRESHOLDS_PATH, classes)
    threshold_vec = thresholds
except Exception as exc:
    threshold_vec = None
    print('Threshold file missing; falling back to scalar threshold when evaluating.', exc)


In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm

data_utils.configure_nih_dataset(img_dir=args['nih_img_dir'],
                                 csv_path=args.get('nih_csv_path'),
                                 train_fraction=args.get('nih_train_fraction', 0.9),
                                 split_seed=args.get('nih_split_seed', 0),
                                 views=['PA'])
_, preprocess = data_utils.get_target_model(args['backbone'], 'cpu')
dataset = data_utils.get_data('nih14_val', preprocess)
loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

weight_matrix = torch.load(Path(MODEL_DIR) / 'W_c.pt')
per_class_scores = torch.zeros_like(weight_matrix)
counts = torch.zeros(weight_matrix.shape[0])

with torch.no_grad():
    for images, _ in tqdm(loader):
        images = images.to(DEVICE)
        _, concept_act = model(images)
        per_class_scores += concept_act.mean(dim=0) * weight_matrix
        counts += 1

per_class_scores /= counts.view(-1, 1)

TOP_K = 15
for idx, name in enumerate(classes):
    row = per_class_scores[idx]
    topk = torch.topk(row, k=TOP_K)
    print(f'Class: {name}')
    for score, concept_idx in zip(topk.values, topk.indices):
        print(f'  {score.item():.4f}	concept {concept_idx.item()}')
    print('-' * 40)
